# SatQuery — serve the pipeline from Kaggle via cloudflared

Kaggle notebooks cannot expose a port, so this runs the FastAPI server on
localhost and puts a cloudflared tunnel in front of it. The printed
`https://*.trycloudflare.com` URL goes into `backend/.env` as
`SATQUERY_VLM_ENDPOINT`.

**Before running:**

1. `Settings → Accelerator → GPU`
2. `Settings → Internet → On` (required for the tunnel and the model download)
3. `+ Add Input` → the training notebook's output (or wherever `classifier.pt` lives)

The tunnel dies when the session ends, so grab a fresh URL each time.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code, dependencies, cloudflared

In [ ]:
# The training code lives in this repo. If it is private, create a GitHub
# personal access token and use:
#   REPO = "https://<TOKEN>@github.com/shubh000015/satquery-ai.git"
# Alternatively upload the ml/ folder as a private Kaggle Dataset and point
# ML_DIR at /kaggle/input/<your-dataset>/ml instead of cloning.
import subprocess, sys, os
from pathlib import Path

REPO = "https://github.com/shubh000015/satquery-ai.git"
CLONE_DIR = Path("/kaggle/working/satquery-ai")

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(CLONE_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)

ML_DIR = CLONE_DIR / "ml"
sys.path.insert(0, str(ML_DIR))
os.chdir(ML_DIR)
print("ml dir:", ML_DIR)
print(sorted(p.name for p in ML_DIR.iterdir()))

In [ ]:
!pip -q install "bitsandbytes>=0.45" "fastapi>=0.115" "uvicorn[standard]>=0.30" 2>&1 | tail -2
!wget -q -O /usr/local/bin/cloudflared \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 2. Find the trained classifier

In [ ]:
from pathlib import Path

found = sorted(Path("/kaggle/input").rglob("classifier.pt")) + \
        sorted(Path("/kaggle/working").rglob("classifier.pt"))
if not found:
    raise SystemExit(
        "classifier.pt not found. Attach the training notebook's output with "
        "'+ Add Input', or re-run kaggle_train_classifier.ipynb."
    )

CHECKPOINT = found[0]
print("checkpoint:", CHECKPOINT)

import json
sidecar = CHECKPOINT.with_suffix(".json")
if sidecar.exists():
    saved = json.load(open(sidecar))
    print("trained metrics:", json.dumps(saved.get("metrics", {}), indent=2)[:600])

## 3. Start the server

`USE_LLM = False` starts in seconds with template phrasing — good for checking
the wiring. `True` pulls Qwen2.5-7B-Instruct in 4-bit (a few minutes on first
run) for the phrased answers.

In [ ]:
import os, subprocess, time

USE_LLM = True
PORT = 8100

os.environ["CHECKPOINT"] = str(CHECKPOINT)
cmd = ["python", "serve.py", "--checkpoint", str(CHECKPOINT), "--port", str(PORT)]
if not USE_LLM:
    cmd.append("--no-llm")

server_log = open("/kaggle/working/serve.log", "wb")
server = subprocess.Popen(cmd, stdout=server_log, stderr=subprocess.STDOUT)
print("server pid", server.pid, "- waiting for it to come up ...")

In [ ]:
# Poll /health until the models finish loading.
import json, time, urllib.request

deadline = time.time() + 900
while time.time() < deadline:
    if server.poll() is not None:
        print(open("/kaggle/working/serve.log").read()[-3000:])
        raise SystemExit("server exited early - see log above")
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as r:
            print(json.dumps(json.load(r), indent=2))
            break
    except Exception:
        time.sleep(10)
else:
    raise SystemExit("server did not become healthy in 15 minutes")

## 4. Local smoke test, before exposing it

In [ ]:
import base64, io, json, urllib.request
from pathlib import Path
from PIL import Image

samples = sorted(Path("/kaggle/input").rglob("images_s2/*.png"))[:1]
if not samples:
    samples = sorted(Path("/kaggle/input").rglob("*.png"))[:1]

buffer = io.BytesIO()
Image.open(samples[0]).convert("RGB").save(buffer, format="PNG")
payload = json.dumps({
    "question": "Is there any water in this image?",
    "imageB64": base64.b64encode(buffer.getvalue()).decode(),
    "task": "vqa",
    "modality": "optical",
}).encode()

request = urllib.request.Request(
    f"http://127.0.0.1:{PORT}/v1/vqa",
    data=payload,
    headers={"Content-Type": "application/json"},
)
with urllib.request.urlopen(request, timeout=180) as response:
    print(json.dumps(json.load(response), indent=2))

## 5. Open the tunnel

In [ ]:
import re, subprocess, time

tunnel_log = "/kaggle/working/cloudflared.log"
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate",
     "--logfile", tunnel_log],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

public_url = None
deadline = time.time() + 120
while time.time() < deadline and public_url is None:
    time.sleep(3)
    try:
        text = open(tunnel_log).read()
    except FileNotFoundError:
        continue
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        public_url = match.group(0)

if not public_url:
    raise SystemExit("could not read the tunnel URL - check " + tunnel_log)

print("PUBLIC URL:", public_url)
print()
print("Put this in backend/.env and restart the backend:")
print(f"  SATQUERY_VLM_ENDPOINT={public_url}")

In [ ]:
# Confirm the tunnel reaches the server from outside.
import json, urllib.request
with urllib.request.urlopen(f"{public_url}/health", timeout=30) as response:
    print(json.dumps(json.load(response), indent=2))

## 6. Keep the session alive

The tunnel lives only as long as this notebook runs. Leave the cell below
running during the demo; interrupt it to shut everything down.

In [ ]:
import time
try:
    while True:
        if server.poll() is not None:
            print("server died:")
            print(open("/kaggle/working/serve.log").read()[-2000:])
            break
        time.sleep(60)
except KeyboardInterrupt:
    print("shutting down")
    tunnel.terminate()
    server.terminate()